In [ ]:
# import os
# from google.colab import drive

# # Mount drive silently if not already mounted
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')

# %run /content/drive/MyDrive/Stocks/configuration.ipynb


In [ ]:
# =========================================================================
# HELPERS

# =========================================================================

def apply_y_scaling(ax):

    ymin, ymax = ax.get_ylim()

    span = ymax - ymin

    raw_step = span / 8.0

    step = max(50, int(np.ceil(raw_step / 50.0)) * 50)

    new_ymin = np.floor(ymin / step) * step

    new_ymax = np.ceil(ymax / step) * step

    if new_ymin == new_ymax:

        new_ymax += step

        new_ymin -= step

    ticks = np.arange(new_ymin, new_ymax + step, step)

    ax.set_ylim(new_ymin, new_ymax)

    ax.set_yticks(ticks)

    ax.set_yticklabels([str(int(t)) for t in ticks])

    for label in ax.get_yticklabels():

        val = int(label.get_text().replace('−', '-'))

        if val >= 0 and (val == int(new_ymin) or val == int(new_ymax)):

            label.set_fontweight('bold')



def get_tag(name):

    match = re.search(r'\[(.*?)\]', name)

    return match.group(1) if match else ""



def format_ret(val):

    formatted = f"+{val:.1f}%" if val >= 0 else f"{val:.1f}%"

    if val >= 100: return f"<b>{formatted}</b>"

    return formatted



def format_range_ret(min_val, max_val):

    formatted_min = f"+{min_val:.1f}%" if min_val >= 0 else f"{min_val:.1f}%"

    formatted_max = f"+{max_val:.1f}%" if max_val >= 0 else f"{max_val:.1f}%"

    if max_val >= 100:

        return f"<b>{formatted_min} to {formatted_max}</b>"

    return f"{formatted_min} to {formatted_max}"



def yf_link(asset_name, ticker=None):

    """Wrap an asset name with a Yahoo Finance link."""

    if ticker is None:

        ticker = single_stocks.get(asset_name, etfs.get(asset_name, ""))

    if not ticker:

        return asset_name

    url = f"https://finance.yahoo.com/quote/{ticker}/"

    return f'<a href="{url}" target="_blank" style="color:#1565C0;">{asset_name}</a>'



# =========================================================================

# DATA FETCHING

# =========================================================================

if not use_cache:

    bulk_df = yf.download(tickers=current_tickers, period="max", group_by="ticker", threads=True, auto_adjust=True)

    for ticker in current_tickers:

        try:

            if len(current_tickers) > 1:

                df_asset = bulk_df.xs(ticker, axis=1, level=0).copy()

            else:

                df_asset = bulk_df.copy()

            df_asset.index = df_asset.index.tz_localize(None)

            df_clean = df_asset.dropna(subset=["Close"])

            if not df_clean.empty:

                name = ticker_to_name[ticker]

                earliest_dates[name] = df_clean.index.min()

                plot_data[name] = df_clean.sort_index()

        except KeyError:

            continue

    with open(CACHE_FILE, "wb") as f:

        pickle.dump({"tickers": set(current_tickers), "plot_data": plot_data, "earliest_dates": earliest_dates}, f)



# =========================================================================

# HTML INITIALIZATION

# =========================================================================

html_elements = [

    "<html><head><style>",

    "body { font-family: Arial, sans-serif; margin: 20px; background-color: #f8f9fa; color: black; }",

    ".report-section { margin-bottom: 40px; background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); overflow-x: auto;}",

    "img { max-width: 100%; height: auto; display: block; margin: 10px 0; }",

    "table { border-collapse: collapse; width: 100%; margin: 15px 0; font-size: 15px; border: 1px solid black; color: black; }",

    "th, td { border: 1px solid black; text-align: center; padding: 8px; color: black; }",

    ".tab { overflow: hidden; border: 1px solid #000; background-color: #f1f1f1; margin-top: 20px; }",

    ".tab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 16px; transition: 0.3s; color: black; font-weight: bold; }",

    ".tab button:hover { background-color: #ddd; }",

    ".tab button.active { background-color: #ccc; }",

    ".tabcontent { display: none; padding: 20px; border: 1px solid #000; border-top: none; background: white; }",

    ".subtab { overflow: hidden; background-color: #e9e9e9; border: 1px solid #000; border-bottom: none; }",

    ".subtab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 10px 14px; transition: 0.3s; color: black; font-size: 14px; }",

    ".subtab button:hover { background-color: #d5d5d5; }",

    ".subtab button.active { background-color: #bbb; }",

    "</style>",

    "<script>",

    "function openTab(evt, tabName, tabClass, contentClass) {",

    "  var i, tabcontent, tablinks;",

    "  tabcontent = document.getElementsByClassName(contentClass);",

    "  for (i = 0; i < tabcontent.length; i++) { tabcontent[i].style.display = 'none'; }",

    "  tablinks = document.getElementsByClassName(tabClass);",

    "  for (i = 0; i < tablinks.length; i++) { tablinks[i].className = tablinks[i].className.replace(' active', ''); }",

    "  document.getElementById(tabName).style.display = 'block';",

    "  if (evt) { evt.currentTarget.className += ' active'; }",

    "}",

    "document.addEventListener('keydown', function(e) {",

    "  if (e.key === 'Tab') {",

    "    e.preventDefault();",

    "    var visibleMain = document.querySelector('.main-tabcontent[style*=\"display: block\"]');",

    "    var visibleSector = document.querySelector('.sector-tabcontent[style*=\"display: block\"]');",

    "    var targetContainer = visibleMain && visibleMain.querySelector('.subtab button.active') ? visibleMain : visibleSector;",

    "    if (targetContainer) {",

    "      var activeSubBtn = targetContainer.querySelector('.subtab button.active');",

    "      if (activeSubBtn) {",

    "        var nextBtn = activeSubBtn.nextElementSibling || activeSubBtn.parentElement.firstElementChild;",

    "        nextBtn.click();",

    "      }",

    "    }",

    "  }",

    "});",

    "</script>",

    "</head><body>",

    '<div class="tab">',

    '<button class="main-tab-links" onclick="openTab(event, \'ETFs\', \'main-tab-links\', \'main-tabcontent\')" id="defaultMainTab">ETFs</button>',

    '<button class="main-tab-links" onclick="openTab(event, \'Stocks\', \'main-tab-links\', \'main-tabcontent\')">Stocks</button>',

    '<button class="main-tab-links" onclick="openTab(event, \'Sector\', \'main-tab-links\', \'main-tabcontent\')">Sector</button>',
    '<button class="main-tab-links" onclick="openTab(event, \'Watchlist\', \'main-tab-links\', \'main-tabcontent\')">Watchlist</button>',

    '</div>'

]



def handle_output(fig, element_id):

    buf = BytesIO()

    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)

    buf.seek(0)

    base64_str = base64.b64encode(buf.read()).decode("utf-8")

    buf.close()

    plt.close(fig)

    html_elements.append(f'<img id="{element_id}" src="data:image/png;base64,{base64_str}"/>')



columns = ["Asset Name", "1Y (%)", "2Y (%)", "5Y (%)", "Risk Config", "Cyclic", "Loss Risk"]

headers_html = "".join([f'<th style="background-color: #2F4F4F; color: white;">{c}</th>' for c in columns])



# =========================================================================

# 1. VISUAL RENDERING LOOP (ETFs and Stocks)

# =========================================================================

for group_name in ["ETFs", "Stocks"]:

    html_elements.append(f'<div id="{group_name}" class="main-tabcontent">')

    html_elements.append('<div class="subtab">')

    for idx, YEARS_TO_PLOT in enumerate(timeframes):

        default_id = f' id="default{group_name}Tab"' if idx == 0 else ""

        html_elements.append(f'<button class="{group_name}-tab-links" onclick="openTab(event, \'{group_name}_{YEARS_TO_PLOT}y\', \'{group_name}-tab-links\', \'{group_name}-tabcontent\')"{default_id}>{YEARS_TO_PLOT}y</button>')



    html_elements.append(f'<button class="{group_name}-tab-links" onclick="openTab(event, \'{group_name}_Table\', \'{group_name}-tab-links\', \'{group_name}-tabcontent\')">Table</button>')

    html_elements.append('</div>')



    for YEARS_TO_PLOT in timeframes:

        html_elements.append(f'<div id="{group_name}_{YEARS_TO_PLOT}y" class="{group_name}-tabcontent">')

        start_window_date = datetime.now() - timedelta(days=int(YEARS_TO_PLOT * 365))

        fig, ax = plt.subplots(figsize=(10.8, 5.4))



        group_tickers = all_groups[group_name]

        valid_names = [k for k in group_tickers.keys() if "[NUC]" not in k and "[QTM]" not in k and "[CYBER]" not in k and "[TECH]" not in k and "[IND]" not in k and "[SPEC]" not in k]

        sorted_names = sorted(valid_names, key=lambda x: (get_tag(x), x))



        vlines_to_draw = []

        current_tag = None



        for name in sorted_names:

            if name not in plot_data: continue

            df = plot_data[name]

            filtered_df = df[df.index >= start_window_date].copy()



            if not filtered_df.empty:

                tag = get_tag(name)

                if current_tag is not None and tag != current_tag:

                    ax.plot([], [], ' ', label=" ")

                current_tag = tag



                base_price = filtered_df["Close"].iloc[0]

                filtered_df["Pct_Increase"] = ((filtered_df["Close"] - base_price) / base_price) * 100

                style = theme_styles.get(name, {"color": "black", "linestyle": "-"})

                ax.plot(filtered_df.index, filtered_df["Pct_Increase"], label=name, linewidth=1.5, color=style["color"], linestyle=style["linestyle"])



                true_start_date = earliest_dates.get(name, start_window_date)

                if true_start_date > start_window_date:

                    vlines_to_draw.append((true_start_date, style["color"]))



        ymin_val, ymax_val = ax.get_ylim()

        for birth_date, matching_color in vlines_to_draw:

            ax.vlines(x=birth_date, ymin=ymin_val, ymax=0, colors=matching_color, linestyles="--", alpha=0.7, linewidth=1.2)



        title_str = group_name[:-1] if group_name.endswith('s') else group_name

        ax.set_title(f"{title_str} ({YEARS_TO_PLOT}y)", fontsize=13, fontweight="bold")

        ax.set_ylabel("Performance (%)", fontsize=11)

        ax.grid(True, linestyle="--", alpha=0.5)

        ax.axhline(0, color="red", linestyle="-", alpha=0.4)

        ax.set_xlim(left=start_window_date, right=datetime.now())



        apply_y_scaling(ax)



        normalized_start = start_window_date.replace(month=1, day=1)

        extended_end = datetime.now() + timedelta(days=31)

        tick_dates = pd.date_range(start=normalized_start, end=extended_end, freq="MS")

        tick_locs = [mdates.date2num(d) for d in tick_dates]



        sparse_labels = []

        for d in tick_dates:

            if YEARS_TO_PLOT <= 0.5:

                sparse_labels.append(d.strftime('%b %Y').upper())

            else:

                if d.month == 1:

                    sparse_labels.append(d.strftime('JAN %Y'))

                elif d.month == 6:

                    sparse_labels.append(d.strftime('JUNE %Y'))

                else:

                    sparse_labels.append("")



        ax.xaxis.set_major_locator(FixedLocator(tick_locs))

        ax.xaxis.set_major_formatter(FixedFormatter(sparse_labels))

        ax.tick_params(axis="x", rotation=35, labelsize=9)



        box = ax.get_position()

        ax.set_position([box.x0, box.y0, box.width * 0.75, box.height])

        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)



        handle_output(fig, f"{group_name.lower()}_{YEARS_TO_PLOT}y")

        html_elements.append('</div>')



    if group_name == "Stocks":

        html_elements.append(f'<div id="{group_name}_Table" class="{group_name}-tabcontent">')

        html_elements.append('<div class="report-section"><table>')

        html_elements.append(f'<tr>{headers_html}</tr>')



        mega_cap_forecasts = {k: v for k, v in stock_forecast_models.items() if "[NUC]" not in k and "[QTM]" not in k and "[CYBER]" not in k and "[TECH]" not in k and "[IND]" not in k and "[SPEC]" not in k}

        for asset_name, model in sorted(mega_cap_forecasts.items()):

            min_growth = model["min_rate"] / 100.0

            max_growth = model["max_rate"] / 100.0



            ret_1y_min = ((1 + min_growth) ** 1 - 1) * 100

            ret_1y_max = ((1 + max_growth) ** 1 - 1) * 100

            ret_2y_min = ((1 + min_growth) ** 2 - 1) * 100

            ret_2y_max = ((1 + max_growth) ** 2 - 1) * 100

            ret_5y_min = ((1 + min_growth) ** 5 - 1) * 100

            ret_5y_max = ((1 + max_growth) ** 5 - 1) * 100



            bg = get_row_bg_color(asset_name)



            cells = [

                yf_link(asset_name),

                format_range_ret(ret_1y_min, ret_1y_max),

                format_range_ret(ret_2y_min, ret_2y_max),

                format_range_ret(ret_5y_min, ret_5y_max),

                model["risk"],

                model["cyclic"],

                model["loss_risk"]

            ]

            cells_html = "".join([f'<td>{cell}</td>' for cell in cells])

            html_elements.append(f'<tr style="background-color: {bg};">{cells_html}</tr>')



        html_elements.append("</table></div></div>")



    elif group_name == "ETFs":

        html_elements.append(f'<div id="{group_name}_Table" class="{group_name}-tabcontent">')

        html_elements.append('<div class="report-section"><table>')

        html_elements.append(f'<tr>{headers_html}</tr>')



        for asset_name, model in sorted(growth_forecast_models.items()):

            annual_growth = model["rate"] / 100.0

            ret_1y = ((1 + annual_growth) ** 1 - 1) * 100

            ret_2y = ((1 + annual_growth) ** 2 - 1) * 100

            ret_5y = ((1 + annual_growth) ** 5 - 1) * 100



            bg = get_row_bg_color(asset_name)



            cells = [

                yf_link(asset_name),

                format_ret(ret_1y),

                format_ret(ret_2y),

                format_ret(ret_5y),

                model["risk"],

                model["cyclic"],

                model["loss_risk"]

            ]

            cells_html = "".join([f'<td>{cell}</td>' for cell in cells])

            html_elements.append(f'<tr style="background-color: {bg};">{cells_html}</tr>')



        html_elements.append("</table></div></div>")



    html_elements.append('</div>')



# =========================================================================

# 2. TAILORED PLOTTING BLOCK (Sector Tab: TECH, QTM, NUC, CYBER + Tables)

# =========================================================================

html_elements.append('<div id="Sector" class="main-tabcontent">')

html_elements.append('<div class="subtab">')

html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_TECH\', \'sector-tab-links\', \'sector-tabcontent\')" id="defaultSectorTab">TECH</button>')

html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_QTM\', \'sector-tab-links\', \'sector-tabcontent\')">QTM</button>')

html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_NUC\', \'sector-tab-links\', \'sector-tabcontent\')">NUC</button>')

html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_CYBER\', \'sector-tab-links\', \'sector-tabcontent\')">CYBER</button>')
html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_IND\', \'sector-tab-links\', \'sector-tabcontent\')">IND</button>')
html_elements.append('<button class="sector-tab-links" onclick="openTab(event, \'Sector_SPEC\', \'sector-tab-links\', \'sector-tabcontent\')">SPEC</button>')

html_elements.append('</div>')



def plot_sector(keyword, title_base, sector_id):

    html_elements.append(f'<div id="{sector_id}" class="sector-tabcontent">')

    html_elements.append('<div class="subtab" style="background-color: #e2e2e2;">')



    for idx, y in enumerate(timeframes):

        def_id = f' id="default_{sector_id}_time"' if idx == 0 else ""

        html_elements.append(f'<button class="{sector_id}-time-links" onclick="openTab(event, \'{sector_id}_{y}y\', \'{sector_id}-time-links\', \'{sector_id}-time-content\')"{def_id}>{y}y</button>')



    html_elements.append(f'<button class="{sector_id}-time-links" onclick="openTab(event, \'{sector_id}_Table\', \'{sector_id}-time-links\', \'{sector_id}-time-content\')">Table</button>')

    html_elements.append('</div>')



    for y in timeframes:

        html_elements.append(f'<div id="{sector_id}_{y}y" class="{sector_id}-time-content">')

        fig, ax = plt.subplots(figsize=(10.8, 5.4))

        universe = {k: v for k, v in single_stocks.items() if keyword in k}



        start_date = datetime.now() - timedelta(days=int(y*365))

        vlines_to_draw = []



        for name in universe.keys():

            if name in plot_data and not plot_data[name].empty:

                df = plot_data[name].copy()

                df = df[df.index >= start_date]

                if df.empty: continue



                df["Pct_Increase"] = ((df["Close"] - df["Close"].iloc[0]) / df["Close"].iloc[0]) * 100

                style = theme_styles.get(name, {"color": "black", "linestyle": "-"})

                ax.plot(df.index, df["Pct_Increase"], label=name, color=style["color"], linestyle=style["linestyle"], linewidth=1.8)



                true_start_date = earliest_dates.get(name, start_date)

                if true_start_date > start_date:

                    vlines_to_draw.append((true_start_date, style["color"]))



        ymin_val, ymax_val = ax.get_ylim()

        for birth_date, matching_color in vlines_to_draw:

            ax.vlines(x=birth_date, ymin=ymin_val, ymax=0, colors=matching_color, linestyles="--", alpha=0.7, linewidth=1.2)



        ax.set_title(f"{title_base} ({y}y)", fontsize=13, fontweight="bold")

        ax.grid(True, linestyle="--", alpha=0.5)

        ax.axhline(0, color="red", linestyle="-", alpha=0.4)

        ax.set_xlim(left=start_date, right=datetime.now())



        apply_y_scaling(ax)



        normalized_start = start_date.replace(month=1, day=1)

        extended_end = datetime.now() + timedelta(days=31)

        tick_dates = pd.date_range(start=normalized_start, end=extended_end, freq="MS")

        tick_locs = [mdates.date2num(d) for d in tick_dates]



        sparse_labels = []

        for d in tick_dates:

            if y <= 0.5:

                sparse_labels.append(d.strftime('%b %Y').upper())

            else:

                if d.month == 1:

                    sparse_labels.append(d.strftime('JAN %Y'))

                elif d.month == 6:

                    sparse_labels.append(d.strftime('JUNE %Y'))

                else:

                    sparse_labels.append("")



        ax.xaxis.set_major_locator(FixedLocator(tick_locs))

        ax.xaxis.set_major_formatter(FixedFormatter(sparse_labels))

        ax.tick_params(axis="x", rotation=35, labelsize=9)



        box = ax.get_position()

        ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])

        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)



        handle_output(fig, f"{sector_id.lower()}_{y}y_img")

        html_elements.append('</div>')



    html_elements.append(f'<div id="{sector_id}_Table" class="{sector_id}-time-content">')

    html_elements.append('<div class="report-section"><table>')

    html_elements.append(f'<tr>{headers_html}</tr>')



    sector_forecasts = {k: v for k, v in stock_forecast_models.items() if keyword in k}

    for asset_name, model in sorted(sector_forecasts.items()):

        min_growth = model["min_rate"] / 100.0

        max_growth = model["max_rate"] / 100.0



        ret_1y_min = ((1 + min_growth) ** 1 - 1) * 100

        ret_1y_max = ((1 + max_growth) ** 1 - 1) * 100

        ret_2y_min = ((1 + min_growth) ** 2 - 1) * 100

        ret_2y_max = ((1 + max_growth) ** 2 - 1) * 100

        ret_5y_min = ((1 + min_growth) ** 5 - 1) * 100

        ret_5y_max = ((1 + max_growth) ** 5 - 1) * 100



        bg = get_row_bg_color(asset_name)



        cells = [

            yf_link(asset_name),

            format_range_ret(ret_1y_min, ret_1y_max),

            format_range_ret(ret_2y_min, ret_2y_max),

            format_range_ret(ret_5y_min, ret_5y_max),

            model["risk"],

            model["cyclic"],

            model["loss_risk"]

        ]

        cells_html = "".join([f'<td>{cell}</td>' for cell in cells])

        html_elements.append(f'<tr style="background-color: {bg};">{cells_html}</tr>')



    html_elements.append("</table></div></div>")

    html_elements.append('</div>')



plot_sector("[TECH]", "Technology", "Sector_TECH")

plot_sector("[QTM]", "Quantum", "Sector_QTM")

plot_sector("[NUC]", "Nuclear", "Sector_NUC")

plot_sector("[CYBER]", "Cybersecurity", "Sector_CYBER")
plot_sector("[IND]", "Industrials & Defense", "Sector_IND")
plot_sector("[SPEC]", "Speculative Growth", "Sector_SPEC")



html_elements.append('</div>')

# =========================================================================
# 3. WATCHLIST TAB (PUMP Dashboard — all fundamental columns)
# =========================================================================
html_elements.append('<div id="Watchlist" class="main-tabcontent">')
html_elements.append('<div class="report-section">')
html_elements.append(f'<h3>Stock Watchlist — Last updated: {LAST_UPDATED}</h3>')

# Build watchlist table with all columns
wl_columns = WATCHLIST_COLUMNS
wl_headers = "".join([f'<th style="background-color: #2F4F4F; color: white; white-space: nowrap; padding: 6px 8px; font-size: 12px;">{col[0]}</th>' for col in wl_columns])
html_elements.append('<table style="font-size: 12px;">')
html_elements.append(f'<tr>{wl_headers}</tr>')

for stock in WATCHLIST_STOCKS:
    bg = get_watchlist_row_color(stock.get("risk", ""))

    # Color-code the pumped status cell
    pumped_val = str(stock.get("pumped", "—"))
    if pumped_val.startswith("YES"):
        pumped_style = "color: #B71C1C; font-weight: bold;"
    elif pumped_val.startswith("PARTIAL"):
        pumped_style = "color: #E65100; font-weight: bold;"
    else:
        pumped_style = "color: #1B5E20; font-weight: bold;"

    cells_html = ""
    for header, key, desc in wl_columns:
        val = str(stock.get(key, "—"))
        if key == "ticker":
            yf_url = f'https://finance.yahoo.com/quote/{val}/'
            cells_html += f'<td style="white-space: nowrap;"><a href="{yf_url}" target="_blank" style="color:#1565C0; font-weight:bold;">{val}</a></td>'
        elif key == "pumped":
            cells_html += f'<td style="white-space: nowrap; {pumped_style}">{val}</td>'
        elif key == "catalyst":
            cells_html += f'<td style="text-align: left; min-width: 200px;">{val}</td>'
        elif key in ("pot_1y", "pot_2y", "pot_5y"):
            cells_html += f'<td style="white-space: nowrap;">{val}</td>'
        else:
            cells_html += f'<td style="white-space: nowrap;">{val}</td>'

    html_elements.append(f'<tr style="background-color: {bg};">{cells_html}</tr>')

html_elements.append('</table>')

# Footnotes
html_elements.append('<div style="margin-top: 15px; font-size: 11px; color: #555;">')
html_elements.append('<b>Pumped Status:</b> ')
html_elements.append('<span style="color: #B71C1C;">YES</span> = Already had a major run | ')
html_elements.append('<span style="color: #E65100;">PARTIAL</span> = Moved significantly, may have more room | ')
html_elements.append('<span style="color: #1B5E20;">NO</span> = Hasn\'t had its breakout move yet')
html_elements.append('<br><b>Abbreviations:</b> Rev TTM = Revenue Trailing 12 Months | Rev Grw = Revenue Growth YoY | ')
html_elements.append('GM = Gross Margin | PM = Profit Margin | FCF M = Free Cash Flow Margin | ')
html_elements.append('PT = Price Target | Pot 1Y/2Y/5Y = Estimated potential price change')
html_elements.append('<br>⚠️ This is NOT financial advice. Data may be outdated. Always do your own due diligence.')
html_elements.append('</div>')

html_elements.append('</div>')
html_elements.append('</div>')




# =========================================================================

# FILE COMPILATION WRITE-OUT & DISPLAY

# =========================================================================

html_elements.append("""

<script>

    document.getElementById('defaultMainTab').click();

    document.getElementById('defaultETFsTab').click();

    document.getElementById('defaultStocksTab').click();

    document.getElementById('defaultSectorTab').click();

    document.getElementById('default_Sector_TECH_time').click();

    document.getElementById('default_Sector_QTM_time').click();

    document.getElementById('default_Sector_NUC_time').click();

    document.getElementById('default_Sector_CYBER_time').click();
    document.getElementById('default_Sector_IND_time').click();
    document.getElementById('default_Sector_SPEC_time').click();


</script>

""")

html_elements.append("</body></html>")



html_content = "\n".join(html_elements)



if HTML:

    os.makedirs(os.path.dirname(HTML_FILE), exist_ok=True)

    with open(HTML_FILE, "w", encoding="utf-8") as html_f:

        html_f.write(html_content)



IPython.display.display(IPython.display.HTML(html_content))









Output hidden; open in https://colab.research.google.com to view.